# Gaussian Mixture Models (GMM) — Solutions Notebook

**Difficulty**: ⭐⭐⭐ Advanced  
**Time**: ~60 mins  
**Complete, verified reference implementation.**

---


## 🎯 Section 1: Overview

A soft-clustering probabilistic model assuming all data points are generated from a mixture of Gaussian distributions.

### GMM Density:
$$p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$


## 🔧 Section 2: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

In [ ]:
from scipy.stats import multivariate_normal

class GMMFromScratch:
    def __init__(self, n_components=3, max_iters=50):
        self.n_components = n_components
        self.max_iters = max_iters
        
    def fit(self, X):
        n_samples, n_features = X.shape
        self.weights = np.ones(self.n_components) / self.n_components
        self.means = X[np.random.choice(n_samples, self.n_components, replace=False)]
        self.covariances = np.array([np.eye(n_features) for _ in range(self.n_components)])
        
        for _ in range(self.max_iters):
            # E-step
            responsibilities = np.zeros((n_samples, self.n_components))
            for k in range(self.n_components):
                responsibilities[:, k] = self.weights[k] * multivariate_normal.pdf(X, mean=self.means[k], cov=self.covariances[k], allow_singular=True)
            responsibilities /= responsibilities.sum(axis=1, keepdims=True)
            
            # M-step
            Nk = responsibilities.sum(axis=0)
            self.weights = Nk / n_samples
            self.means = np.dot(responsibilities.T, X) / Nk[:, np.newaxis]
            for k in range(self.n_components):
                diff = X - self.means[k]
                self.covariances[k] = np.dot(responsibilities[:, k] * diff.T, diff) / Nk[k]
        return self


In [ ]:
# Verify that the implementation runs and outputs correctly
X = np.random.rand(50, 2)
model = GMMFromScratch(n_components=2)
model.fit(X)
print('Means:\n', model.means)


## 📦 Section 3: Library Implementation


In [ ]:
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=3, covariance_type='full')
gmm.fit(X)
labels = gmm.predict(X)


## ❓ Section 4: Interview Questions


### Q1: What is the difference between hard and soft clustering?
**Answer**: Hard clustering assigns each point to one cluster exactly. Soft clustering assigns a probability distribution of belonging to each cluster (e.g. responsibilities in GMM).


### Q2: What algorithm optimizes GMM parameters?
**Answer**: The Expectation-Maximization (EM) algorithm, which alternates between estimating assignments (E-step) and updating distribution parameters (M-step).
